In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import copy
from tqdm import tqdm
from pathlib import Path
import networkx as nx

from src.load_data import (
    read_graph_transport_networks_tntp,
    read_traffic_mat_transport_networks_tntp,
)

from src.models import BeckmannModel, TwostageModel
from src.algs import cyclic, ustm, frank_wolfe, N_conjugate_frank_wolfe
from src.salim import SaddleOracle, combined_salim
from src.saddle_ta import salim_ta, chambolle_pock_ta
from src.approx import seq_quad

import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator


plt.rcParams.update({'font.size': 14})
%config InlineBackend.figure_format = 'retina'

%matplotlib inline

# Load network data

In [3]:
networks_path = Path("./TransportationNetworks")

folder = "SiouxFalls"
net_name = "SiouxFalls_net"
traffic_mat_name = "SiouxFalls_trips"

# folder = "Anaheim"
# net_name = "Anaheim_net"
# traffic_mat_name = "Anaheim_trips"

# folder = "Barcelona"
# net_name = "Barcelona_net"
# traffic_mat_name = "Barcelona_trips"

# folder = "Austin"
# net_name = "Austin_net"
# traffic_mat_name = "Austin_trips_am"

# folder = "Berlin-Friedrichshain"
# net_name = "friedrichshain-center_net"
# traffic_mat_name = "friedrichshain-center_trips"

# folder = "Terrassa-Asymmetric"
# net_name = "Terrassa-Asym_net"
# traffic_mat_name = "Terrassa-Asym_trips"

net_file = networks_path / folder / f"{net_name}.tntp"
traffic_mat_file = networks_path / folder / f"{traffic_mat_name}.tntp"
graph, metadata = read_graph_transport_networks_tntp(net_file)
correspondences = read_traffic_mat_transport_networks_tntp(traffic_mat_file, metadata)
n = graph.number_of_nodes()

print(f"{graph.number_of_edges()=}, {graph.number_of_nodes()=}")

metadata["can_pass_through_zones"]=True
graph.number_of_edges()=76, graph.number_of_nodes()=24


In [4]:
traffic_mat = correspondences.traffic_mat.copy()
departures, arrivals = traffic_mat.sum(axis=1), traffic_mat.sum(axis=0)
l, w = departures, arrivals

# Create instances of models/oracles

In [5]:
mu_bm=None
beckmann_model = BeckmannModel(graph, copy.deepcopy(correspondences),mu_bm)
# twostage_beckmann_model = TwostageModel(beckmann_model, departures=departures, arrivals=arrivals, gamma=0.1)
# saddle_oracle = SaddleOracle(twostage_beckmann_model.traffic_model, twostage_beckmann_model.gamma, l, w)

eps = 1e-6
mean_bw = beckmann_model.graph.ep.capacities.a.mean()
mean_cost = beckmann_model.graph.ep.free_flow_times.a.mean()

# cost suboptimality <= eps * (average link cost * avg bandwidth * |E| \approx total cost when beta=1)
eps_abs = eps * mean_cost * mean_bw * graph.number_of_edges()

eps_cons_abs = eps * mean_bw
# sum of capacity violation <= eps * average link capacity
print(eps_abs, eps_cons_abs)

3.217622777832031 0.010247206298828124


In [13]:
# or just call 
# beckmann_model.solve_cvxpy(solver="CLARABEL", verbose=True)

import cvxpy as cp
traffic_lapl = SaddleOracle(beckmann_model, None, None, None).Bmul(beckmann_model.correspondences.traffic_mat).T

incidence_mat = nx.incidence_matrix(graph, oriented=True).todense()

capacities = np.array(list(nx.get_edge_attributes(graph, "capacities").values()), dtype=np.float32)
ffts = np.array(
    list(nx.get_edge_attributes(graph, "free_flow_times").values()),
    dtype=np.float32,
)
rhos = np.array(list(nx.get_edge_attributes(graph, "rho").values()), dtype=np.float32)
mus = np.array(list(nx.get_edge_attributes(graph, "mu").values()), dtype=np.float32)
mus_inv = np.round(1 / mus).astype(int)

assert np.allclose(mus_inv, 1 / mus), "now only works for integer 1/mu s"


flows_ei = cp.Variable((len(graph.edges), traffic_lapl.shape[1]), nonneg=True)
flows_e = cp.sum(flows_ei, axis=1)

# in cvxpy, power should be scalar, so use loop
sigmas = [
    ffts[e]
    * (
        flows_e[e]
        + (rhos[e] / (1 + mus_inv[e])) * (cp.pos(flows_e[e]) ** int(1 + mus_inv[e]) / capacities[e] ** int(mus_inv[e]))

    )
    for e in range(len(graph.edges))
]
objective = cp.Minimize(cp.sum(sigmas))

prob = cp.Problem(
    objective,
    [
        (incidence_mat @ flows_ei) == -traffic_lapl,
    ],
)
prob.solve(solver="CLARABEL", verbose=True, tol_feas=1e3)
flows_ei = flows_ei.value if flows_ei is not None else None
potentials = [cons.dual_value for cons in prob.constraints]
flows_ei, potentials



(CVXPY) Dec 15 05:18:32 PM: Your problem has 1824 variables, 576 constraints, and 0 parameters.
(CVXPY) Dec 15 05:18:32 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Dec 15 05:18:32 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Dec 15 05:18:32 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Dec 15 05:18:32 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Dec 15 05:18:32 PM: Compiling problem (target solver=CLARABEL).
(CVXPY) Dec 15 05:18:32 PM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> CLARABEL
(CVXPY) Dec 15 05:18:32 PM: Applying reduction Dcp2Cone
(CVXPY) Dec 15 05:18:32 PM: Applying reduction CvxAttr2Constr
(CVXPY) Dec 15 05:18:32 PM: Applying reduction ConeMatrixStuffing


                                     CVXPY                                     
                                     v1.7.2                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Dec 15 05:18:32 PM: Applying reduction CLARABEL
(CVXPY) Dec 15 05:18:32 PM: Finished problem compilation (took 5.752e-01 seconds).
(CVXPY) Dec 15 05:18:32 PM: Invoking solver CLARABEL  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
-------------------------------------------------------------
           Clarabel.rs v0.11.1  -  Clever Acronym                

                   (c) Paul Goulart                          
                University of Oxford, 2022                   
-------------------------------------------------------------

problem:
  variables     = 2128
  constraints   = 3236
  nnz(P)        = 0
  nnz(A)        = 8436
  cones (total) = 230
    :        Zero = 1,  numel = 576
    : Nonnegative = 1,  numel = 1976
    : SecondOrder = 228,  numel = (3,3,3,3,...,3)

settings:
  linear algebra: direct / qdldl, precision: 64 bit (1 thread)
  max iter = 200, time limit = Inf,  max step = 0.990
  tol_feas = 1.0e3, tol_gap_abs = 1.0e-8, tol_gap_rel = 1.0e-8,
  static 

(CVXPY) Dec 15 05:18:32 PM: Problem status: infeasible
(CVXPY) Dec 15 05:18:32 PM: Optimal value: inf
(CVXPY) Dec 15 05:18:32 PM: Compilation took 5.752e-01 seconds
(CVXPY) Dec 15 05:18:32 PM: Solver (including time spent in interface) took 2.921e-02 seconds


  0  -2.5921e-11  -9.4603e+02  9.46e+02  7.41e-01  1.19e-02  1.00e+00  5.59e+04   ------   
  1  +3.1136e+06  +3.2284e+06  3.69e-02  7.23e-01  1.04e-02  1.17e+05  2.61e+04  9.44e-01  
  2  +4.7085e+07  +7.2860e+07  5.47e-01  3.17e-01  1.73e-03  2.58e+07  7.23e+03  8.51e-01  
  3  +2.5484e+07  +1.0508e+08  3.12e+00  4.32e-02  1.96e-04  7.96e+07  1.11e+03  8.64e-01  
  4  +1.1854e+07  +1.5267e+08  1.19e+01  1.77e-02  7.89e-05  1.41e+08  6.25e+02  7.23e-01  
  5  +1.5765e+07  +7.7702e+08  4.83e+01  3.55e-03  1.57e-05  7.61e+08  1.24e+02  9.66e-01  
  6  +2.1742e+07  +2.4852e+09  1.13e+02  7.35e-04  3.26e-06  2.46e+09  4.08e+01  9.18e-01  
---------------------------------------------------------------------------------------------
Terminated with status = PrimalInfeasible
solve time = 23.585646ms
-------------------------------------------------------------------------------
                                    Summary                                    
-----------------------------------

(None, [None])

In [7]:
flows_feas = seq_quad(beckmann_model, iters=1, solution_flows=np.zeros(beckmann_model.graph.num_edges()), 
                      return_full=True, need_log=False)

(24, 76) (24, 24)


In [8]:
flows_feas = np.maximum(0, flows_feas)
np.linalg.norm(incidence_mat @ flows_feas + traffic_lapl), np.all(flows_feas >= 0)

(np.float64(2.5660308270895656e-05), np.True_)